# Spark Streaming with Dummy Kafka - Complete Demo
## End-to-End Example with In-Memory Data Generation

In [1]:
%%capture
# Install required libraries
!pip install kafka-python pyspark findspark

<class 'OSError'>: Not available

In [ ]:
import json
import time
from threading import Thread
from kafka import KafkaProducer, KafkaConsumer
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import findspark
findspark.init()

## 1. Create Dummy Kafka Cluster in Memory

In [ ]:
# In-memory Kafka setup
class DummyKafka:
    def __init__(self):
        self.topic = 'iot_stream'
        self.messages = []

    def produce(self):
        """Generate dummy IoT device data"""
        devices = [f'device_{i}' for i in range(1, 6)]
        while True:
            data = {
                'timestamp': str(time.strftime('%Y-%m-%d %H:%M:%S')),
                'device_id': devices[time.localtime().tm_sec % 5],
                'temperature': round(20 + (time.time() % 10), 1)
            }
            self.messages.append(json.dumps(data))
            time.sleep(1)

    def consume(self):
        """Simulate Kafka consumer"""
        while True:
            if self.messages:
                yield self.messages.pop(0)
            time.sleep(0.1)

# Start dummy Kafka
dummy_kafka = DummyKafka()
Thread(target=dummy_kafka.produce, daemon=True).start()

## 2. Spark Streaming Setup

In [ ]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("KafkaStreamDemo") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

# Define schema for our IoT data
schema = StructType([
    StructField("timestamp", TimestampType()),
    StructField("device_id", StringType()),
    StructField("temperature", DoubleType())
])

## 3. Streaming Pipeline

In [ ]:
# Read from dummy Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "dummy:9092") \
    .option("subscribe", "iot_stream") \
    .option("startingOffsets", "latest") \
    .load()

# Convert binary Kafka value to string
json_df = df.select(
    from_json(col("value").cast("string"), schema).alias("data")

# Process data
processed_df = json_df.select("data.*") \
    .withWatermark("timestamp", "2 minutes") \
    .groupBy(
        window("timestamp", "1 minute", "30 seconds"),
        "device_id"
    ).agg(
        avg("temperature").alias("avg_temp"),
        count("*").alias("reading_count")
    )

# Output to console
query = processed_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()

## 4. Run & Monitor Stream

In [ ]:
# Let stream run for 60 seconds
try:
    time.sleep(60)
except KeyboardInterrupt:
    pass
finally:
    query.stop()
    spark.stop()
    print("Stream stopped")

## 5. Code Explanation

**1. Dummy Kafka Setup**
- Creates in-memory message queue
- Generates IoT device data every second
- Devices: device_1 to device_5
- Random temperatures between 20-30°C

**2. Spark Config**
- `spark.sql.shuffle.partitions`: Controls parallelism
- Schema: Defines structure of JSON data

**3. Streaming Pipeline**
- `withWatermark`: Handles late-arriving data (2 min threshold)
- Tumbling window: 1 minute window sliding every 30 seconds
- Aggregations: Average temperature & reading count per device

**4. Output**
- Console sink shows updates every trigger
- Output modes:
  - **Update**: Only changed results
  - **Complete**: Full aggregation
  - **Append**: New rows only

## 6. Expected Output
```
+------------------------------------------+---------+------------------+--------------+
|window                                    |device_id|avg_temp          |reading_count |
+------------------------------------------+---------+------------------+--------------+
|{2023-09-15 14:25:00, 2023-09-15 14:26:00}|device_2 |24.356            |12            |
|{2023-09-15 14:25:30, 2023-09-15 14:26:30}|device_5 |23.891            |9             |
+------------------------------------------+---------+------------------+--------------+
```

**Key Observations:**
- Windows update every 30 seconds
- Each device shows different counts based on data frequency
- Temperature averages calculated per window

## 7. Teaching Points

**Micro-Batch Execution**
- Spark processes data in 30-second intervals (slide duration)
- Each batch contains 1-minute window of data

**Watermarking**
- State kept for 2 minutes past event time
- Automatic cleanup of old state

**Fault Tolerance**
- Checkpointing (not shown) can be added
- Exactly-once processing guarantees

**Scaling to Real Kafka**
1. Replace dummy Kafka with:
```python
.option("kafka.bootstrap.servers", "real-kafka:9092")
```
2. Add authentication if required
3. Adjust partitions for throughput